In [3]:
import pandas as pd
import gradio as gr
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("ipl.csv", low_memory=False)

df = df[df['innings'] == 1]
df = df.dropna(subset=['team_runs', 'team_wicket', 'overs'])

# =========================
# FEATURE ENGINEERING
# =========================
df['balls'] = df['overs'] * 6
df['run_rate'] = df['team_runs'] / (df['overs'] + 0.1)
df['balls_left'] = 120 - df['balls']
df['wickets_left'] = 10 - df['team_wicket']

# =========================
# FEATURES & TARGET
# =========================
X = df[['team_runs', 'team_wicket', 'overs', 'run_rate', 'balls_left', 'wickets_left']]

y = df['team_runs'] + (df['run_rate'] * (df['balls_left'] / 6))
# =========================
# TRAIN MODEL
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=200
)

model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print("🔥 Model Accuracy:", accuracy)

# =========================
# ANALYTICS FUNCTIONS
# =========================
def score_range(pred):
    low = int(pred - 10)
    high = int(pred + 10)
    return low, int(pred), high
def win_probability(pred_score, current_score, wickets):
    remaining = pred_score - current_score
    wicket_factor = (10 - wickets) / 10
    overs_factor = max(0.1, min(1, remaining / pred_score))
    prob = (1 - overs_factor) * wicket_factor * 100
    return round(min(max(prob, 5), 95), 2)


def create_projection(current_runs, predicted_score):
    overs = np.arange(0, 21, 1)
    curve = current_runs + (predicted_score - current_runs) * (overs / 20) ** 1.2

    plt.figure(figsize=(6, 4))
    plt.plot(overs, curve, marker='o')
    plt.title("📈 IPL Score Projection")
    plt.xlabel("Overs")
    plt.ylabel("Runs")
    plt.grid(True)

    return plt

# =========================
# MAIN PREDICTION FUNCTION
# =========================
def predict(team_bat, team_bowl, venue, runs, wickets, overs):

    balls = overs * 6
    run_rate = runs / (overs + 0.1)
    balls_left = 120 - balls
    wickets_left = 10 - wickets
    pred = model.predict([[runs, wickets, overs, run_rate, balls_left, wickets_left]])[0]

    low, mid, high = score_range(pred)
    win_prob = win_probability(pred, runs, wickets)
    graph = create_projection(runs, pred)

    return (
f"""
🏏 IPL LIVE ANALYTICS DASHBOARD

━━━━━━━━━━━━━━━━━━━━
🟣 Batting Team: {team_bat}
🔴 Bowling Team: {team_bowl}
🏟️ Venue: {venue}

━━━━━━━━━━━━━━━━━━━━
🔥 Current Score: {runs}/{wickets}
⏱️ Overs: {overs}

📊 PREDICTED SCORE RANGE:
🔻 Low: {low}
🎯 Expected: {mid}
🔺 High: {high}

🎯 WIN PROBABILITY:
👉 {win_prob}%

━━━━━━━━━━━━━━━━━━━━
⚡ MODEL ACCURACY:
👉 {round(accuracy*100, 2)}%

🚀 Status: AI Model Active
""",
        graph
    )

# =========================
# UI DESIGN (PRO DASHBOARD)
# =========================
teams = ["CSK", "MI", "RCB", "KKR", "SRH", "RR", "DC", "PBKS"]
venues = ["Wankhede", "Chepauk", "Eden Gardens", "Chinnaswamy", "Narendra Modi Stadium"]

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🏆 IPL AI PRO DASHBOARD
    ### 🔥 Clean + Accurate Score Prediction Engine
    """)

    with gr.Row():
        team_bat = gr.Dropdown(teams, label="🏏 Batting Team")
        team_bowl = gr.Dropdown(teams, label="🎯 Bowling Team")
        venue = gr.Dropdown(venues, label="🏟️ Venue")

    with gr.Row():
        runs = gr.Number(label="🏃 Current Runs")
        wickets = gr.Number(label="🚩 Wickets")
        overs = gr.Number(label="⏱️ Overs")

    btn = gr.Button("🚀 Predict Score", variant="primary")

    output_text = gr.Textbox(label="📊 Match Analysis", lines=14)
    graph_output = gr.Plot(label="📈 Score Projection")

    btn.click(
        predict,
        inputs=[
            team_bat,
            team_bowl,
            venue,
            runs,
            wickets,
            overs
        ],
        outputs=[output_text, graph_output]
    )

demo.launch()

🔥 Model Accuracy: 0.999641274292443


C:\Users\Ashwini\AppData\Local\Temp\ipykernel_3528\1425551594.py:128: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


C:\Users\Ashwini\anaconda3\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
